# COVID-19 Laboratory Data Analysis

This notebook reproduces the exploratory data analysis and statistical testing workflow for the COVID-19 laboratory dataset.

In [ ]:
import math
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

%matplotlib inline
sns.set_theme(style="whitegrid")

notebook_dir = Path.cwd()
candidate_paths = [
    notebook_dir / "dataset.xlsx",
    notebook_dir / "COVID-19-Laboratory-Data-Analysis" / "dataset.xlsx",
    Path('/content/drive/MyDrive/dataset.xlsx'),
]

file_path = next((p for p in candidate_paths if p.exists()), None)
if file_path is None:
    raise FileNotFoundError("dataset.xlsx not found. Place it in the project folder or update the path.")

data = pd.read_excel(file_path)
df = data.copy()

df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes.value_counts()

In [ ]:
(df.isna().mean() * 100).sort_values(ascending=False)

In [ ]:
df = df.loc[:, df.isna().mean() < 0.9]
df = df.drop(columns=["Patient ID"])

df.shape

In [ ]:
df["SARS-Cov-2 exam result"].value_counts(normalize=True) * 100

In [ ]:
float_cols = df.select_dtypes(include="float").columns
object_cols = df.select_dtypes(include="object").columns

num_float_cols = len(float_cols)
plots_per_row = 4

for i in range(0, num_float_cols, plots_per_row):
    current_cols = float_cols[i : i + plots_per_row]
    num_current_plots = len(current_cols)

    fig, axes = plt.subplots(1, num_current_plots, figsize=(4 * num_current_plots, 4))

    if num_current_plots == 1:
        axes = [axes]

    for j, col in enumerate(current_cols):
        sns.histplot(df[col].dropna(), kde=True, ax=axes[j])
        axes[j].set_title(col)

    plt.tight_layout()
    plt.show()

In [ ]:
cols = float_cols
n_cols = 7
n_rows = math.ceil(len(cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(cols):
    sns.histplot(
        df[df["SARS-Cov-2 exam result"] == "positive"][col].dropna(),
        color="green",
        label="Positive",
        kde=True,
        ax=axes[i],
        stat="density",
        alpha=0.5,
    )
    sns.histplot(
        df[df["SARS-Cov-2 exam result"] == "negative"][col].dropna(),
        color="red",
        label="Negative",
        kde=True,
        ax=axes[i],
        stat="density",
        alpha=0.5,
    )
    axes[i].set_title(col)
    axes[i].legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(df["Influenza A"], df["SARS-Cov-2 exam result"])

In [ ]:
df["est_malade"] = df[object_cols].notna().any(axis=1)
pd.crosstab(df["est_malade"], df["SARS-Cov-2 exam result"])

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df[float_cols].corr(), cmap="coolwarm")
plt.show()

In [ ]:
significant_vars = []

for col in float_cols:
    pos = df[df["SARS-Cov-2 exam result"] == "positive"][col].dropna()
    neg = df[df["SARS-Cov-2 exam result"] == "negative"][col].dropna()
    stat, p = ttest_ind(pos, neg, equal_var=False)
    if p < 0.05:
        significant_vars.append(col)

significant_vars